In [ ]:
# Install dependencies
!pip uninstall -y -q torch torchvision torchaudio cudf easyocr numba-cuda dask-cuda pylibraft-cu12 raft-dask-cu12 pylibcudf-cu12 rmm-cu12 quantecon cuvs-cu12 ucxx-cu12 cuml-cu12 cuda-python cuml libcuml libcuvs libraft numba ydata-profiling google-colab
!pip install -q torch==2.5.1+cu121 torchvision==0.20.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install -q torch>=2.1.0 transformers>=5.15.0 datasets>=2.14.0 numpy>=1.24.0 matplotlib>=3.7.0 scipy>=1.10.0 papermill>=2.7.0 testbook>=0.4.2 diptest>=0.11.0
!pip install -q flash-linear-attention==0.4.2 --no-build-isolation
!pip install -q transformers==5.15.1


In [ ]:
import os
import sys
import time
import traceback
import torch
# Fix allocator fragmentation (exp16r OOM: reserved-but-unallocated
# grew to 1.1GB across alphas) — set BEFORE any CUDA use.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# KAGGLEHUB_CACHE: persist models across sessions
os.environ["KAGGLEHUB_CACHE"] = "/kaggle/working/.model_cache"
os.makedirs("/kaggle/working/.model_cache", exist_ok=True)
WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.expanduser("~/.kaggle_test")
sys.path.insert(0, WORKING)
num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    device_L, device_S = "cuda:0", "cuda:1"
    dtype_L = dtype_S = torch.float32
elif num_gpus == 1:
    device_L = device_S = "cuda:0"
    dtype_L = torch.float16
    dtype_S = torch.float32
else:
    device_L = device_S = "cpu"
    dtype_L = dtype_S = torch.float32
if num_gpus > 0:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"torch {torch.__version__} | L={device_L}, S={device_S}")
RES = os.path.join(WORKING, "results")
os.makedirs(RES, exist_ok=True)
# Resolve models: kernel_sources (prev commit outputs) -> working -> HF
def resolve_model(hf_id):
    model_dir = "qwen35-08b" if "0.8B" in hf_id else "qwen35-4b"
    # 1. Check each slug in /kaggle/input/ — listdir only (isdir lies on mounts)
    input_dir = "/kaggle/input"
    try:
        slugs = os.listdir(input_dir)
        print(f"resolve {hf_id}: slugs={slugs}")
        for slug in slugs:
            p_sub = os.path.join(input_dir, slug)
            try:
                contents = os.listdir(p_sub)
            except OSError:
                continue
            if model_dir in contents:
                p = os.path.join(p_sub, model_dir)
                print(f"Found {hf_id} at {p} (via {slug})")
                return p
    except Exception as e:
        print(f"DEBUG: /kaggle/input/ check failed: {e}")
    # 2. Check WORKING
    wf = os.path.join(WORKING, model_dir)
    if os.path.isdir(wf):
        print(f"Found {hf_id} at {wf}")
        return wf
    # 3. Return HF ID - will download + persist
    print(f"Need to download {hf_id} from HF")
    return hf_id
# Load + save for persistence (calls from_pretrained, then save_pretrained if from HF)
def load_model(cls, hf_id, **kw):
    path = resolve_model(hf_id)
    model = cls.from_pretrained(path, **kw)
    # Persist if loaded from HF
    if path == hf_id:
        dst = os.path.join(WORKING, "qwen35-08b" if "0.8B" in hf_id else "qwen35-4b")
        if not os.path.isdir(dst):
            print(f"Saving {hf_id} to {dst}...")
            model.save_pretrained(dst)
            ensure_tokenizer(hf_id, dst)
            print(f"Saved")
    return model
# Ensure tokenizer files exist in a persisted model dir
def ensure_tokenizer(hf_id, dst):
    import transformers
    if not os.path.exists(os.path.join(dst, "tokenizer.json")):
        print(f"Saving tokenizer for {hf_id}...")
        transformers.AutoTokenizer.from_pretrained(hf_id).save_pretrained(dst)
SMALL = resolve_model("Qwen/Qwen3.5-0.8B")
LARGE = resolve_model("Qwen/Qwen3.5-4B")


In [ ]:
import contextlib
import logging
import torch
from transformers import DynamicCache
from transformers.cache_utils import (  # type: ignore[import-not-found]
    DynamicLayer,
    LinearAttentionLayer,
)
def compute_jacobian_for_tokens(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
    prefix_cache=None,
    h_override: torch.Tensor | None = None,
) -> torch.Tensor:
    """Compute Jacobian of logits w.r.t. hidden state at layer_idx.
    Args:
        model: Qwen3_5ForCausalLM instance.
        layer_idx: Layer index (0-based) to compute Jacobian at.
        input_ids: [1, S] token IDs.
        attention_mask: [1, S] attention mask.
        token_set: list of token IDs to compute gradients for.
        prefix_cache: optional pre-computed prefix cache (K/V for positions 0..S-2).
        h_override: if given, used as the hidden state of the last position at
            layer_idx INSTEAD of extracting it from a full forward. The prefix
            cache is still built from input_ids[:, :-1] (context unchanged); the
            caller is responsible for h_override being consistent with
            input_ids (e.g. h_override from the same sequence). Shapes: [1, 1, D],
            [1, D] or [D]. Device must match input_ids.device (no auto-move).
            dtype is cast to the model dtype. The tensor is cloned and detached
            (fresh leaf; the caller's tensor is never mutated).
    Returns:
        J: [K, D] Jacobian matrix (K = len(token_set), unnormalized).
    Interaction with compute_hidden_jacobian:
        compute_jacobian_for_tokens takes `h_override`; compute_hidden_jacobian
        takes `h`. Both mean "use this h instead of computing it from the
        forward", but the names differ for historical reasons (compute_hidden_
        jacobian already had `h`). With h_override=None the behavior is
        bit-identical to pre-v6.0.
    """
    model.eval()
    S = input_ids.shape[1]
    # Constraint: single sequence, no padding. The replay phase builds causal
    # masks from the cache length with attention_mask=None, so any padding
    # token in the prefix would be silently treated as valid context.
    if attention_mask is not None and not bool(attention_mask.all()):
        raise ValueError(
            "compute_jacobian_for_tokens: only unpadded single sequences are "
            "supported (the replay phase cannot honor a padding mask)."
        )
    if prefix_cache is None:
        prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    if h_override is None:
        h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)
        h = h.to(next(model.parameters()).dtype).detach().requires_grad_(True)
    else:
        h = _normalize_h_override(h_override, model, input_ids)
    # Re-enable grad: the decoder forward must build a graph over the leaf
    # h AND the row loop below must run autograd.grad with grad mode ON,
    # even when the caller is inside torch.no_grad() (e.g. eval loops in
    # exp18/19). A no_grad caller otherwise gets "element 0 ... does not
    # have a grad_fn" — the rows loop is part of the same enable_grad block.
    with torch.enable_grad():
        logits = _run_decoder_pass(model, prefix_cache, h, S, layer_idx)
        J = _compute_jacobian_rows(logits, h, token_set)
    return J
def _normalize_h_override(
    h_override: torch.Tensor, model, input_ids: torch.Tensor
) -> torch.Tensor:
    """Validate + normalize h_override to a detached [1, 1, D] leaf.
    Checks (in order, before any compute): shape reducible to [1, 1, D],
    D == model hidden_size, device == input_ids.device. dtype is cast to the
    model dtype. The tensor is cloned+detached (fresh leaf; caller's tensor
    never mutated, no parasitic CopyBackwards nodes).
    """
    if h_override.dim() not in (1, 2, 3) or h_override.numel() == 0:
        raise ValueError(
            f"h_override: shape {tuple(h_override.shape)} is not reducible "
            "to [1, 1, D] (accepted: [D], [1, D], [1, 1, D])"
        )
    if any(s != 1 for s in h_override.shape[:-1]):
        raise ValueError(
            f"h_override: shape {tuple(h_override.shape)} is not reducible "
            "to [1, 1, D] (accepted: [D], [1, D], [1, 1, D])"
        )
    h = h_override.reshape(1, 1, -1)
    # D lookup: flat CausalLM has config.hidden_size; the multimodal wrapper
    # (Qwen3_5ForConditionalGeneration) exposes it on config.text_config only
    # (the outer config is a heterogeneity spec without hidden_size).
    D_expected = None
    cfg = model.config
    for cand in (getattr(cfg, "text_config", None), cfg):
        if cand is not None and hasattr(cand, "hidden_size"):
            D_expected = cand.hidden_size
            break
    if D_expected is None:
        raise ValueError(
            "h_override: cannot resolve model hidden_size "
            "(neither config nor config.text_config has it)"
        )
    D = h.shape[-1]
    if D != D_expected:
        raise ValueError(
            f"h_override: D={D} does not match model hidden_size={D_expected}"
        )
    if h.device != input_ids.device:
        raise ValueError(
            f"h_override: device {h.device} does not match "
            f"input_ids.device {input_ids.device} (no auto-move)"
        )
    model_dtype = next(model.parameters()).dtype
    # clone first, then cast: detach/clone act on the caller's values before
    # any cast, and `to(same dtype)` is a no-op, so the typical fp32 case
    # allocates exactly once (audit docs/changed_review.md, item 3).
    return h.detach().clone().to(model_dtype).requires_grad_(True)
def _run_prefix_pass(
    model, input_ids: torch.Tensor, attention_mask: torch.Tensor
) -> DynamicCache:
    """Prefix pass: all tokens except last -> cache."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids[:, :-1],
            attention_mask=attention_mask[:, :-1],
            use_cache=True,
            output_hidden_states=False,
        )
    return outputs.past_key_values
def _extract_hidden_state(model, input_ids, attention_mask, layer_idx) -> torch.Tensor:
    """Full pass: extract hidden state at layer_idx for last token."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False,
        )
    # hidden_states[0] = embeddings, [l+1] = output of layer l
    h = outputs.hidden_states[layer_idx + 1][:, -1:, :]
    return h  # [1, 1, D]
def _deep_clone_cache(cache):
    """Deep clone cache, ensuring all tensors are independent copies.
    Must copy every per-layer state field, not just the obvious tensors. Both
    layer types swallow unknown constructor kwargs, so passing
    ``keys=`` / ``conv_states=`` / ``recurrent_states=`` silently discards the
    state and yields an empty cache. We therefore assign attributes directly:
    * DynamicLayer (full attention): keys / values / is_initialized / dtype /
      device. Without this, ``update()`` lazy-initialises an empty cache and
      each layer attends only to the current token.
    * LinearAttentionLayer (GDN): conv_states / recurrent_states plus the flag
      dicts (``has_previous_state`` etc.). Without these, GDN layers treat the
      new token as the first in the sequence and drop all history.
    All cloned state tensors are .detach()-ed: the prefix cache is pure data
    (built under no_grad). Detaching drops pointless CopyBackwards graph nodes
    in the enable_grad decode. The transformers 5.15 in-place cache updates
    (update_recurrent_state's copy_, causal_conv1d_update's state update —
    cudagraph static-address optimisation) are NOT fixed by the detach —
    the in-place copy_ lands on the kernel's saved input regardless. Those
    are handled by _clone_gdn_states_on_entry (clone at kernel entry so the
    read object and the in-place-written object are different objects).
    Extra unknown attributes are deep-copied as a forward-compat safety net.
    Cache OBJECT-level attributes (e.g. ``offloading`` on 5.15.1; 4.x carried
    ``_seen_tokens``) are copied from ``vars(cache)`` after the layer loop:
    a fresh ``DynamicCache()`` would reset them. On 5.15.1 ``get_seq_length()``
    is derived from ``keys.shape`` (not an object field), so the post-clone
    assert below passes on both paths — on a version that tracks sequence
    length in cache state it turns a silent Jacobian corruption into a loud
    error.
    ``copy.deepcopy(prefix_cache)`` DOES produce independent copies on 5.15.1
    (tensor values are not shared). _deep_clone_cache exists for (a) detached
    tensors (no spurious CopyBackwards in the enable_grad decode) and
    (b) robustness to per-layer attribute renames — not because deepcopy
    shares storage. Neither clone fixes the in-place write into a graph-saved
    node; that is _clone_gdn_states_on_entry's job (see its docstring).
    """
    import copy as _copy
    def _cl(t):
        return t.clone().detach() if t is not None else None
    new_cache = DynamicCache()
    for idx in range(len(cache.layers)):
        layer = cache.layers[idx]
        if isinstance(layer, DynamicLayer):
            # Deliberately NOT a vars()-walk: full-attention state fields are
            # stable across versions, and a rename would fail LOUD
            # (AttributeError), not silently corrupt the decode. The GDN
            # branch below IS generic, because GDN state fields have
            # demonstrably drifted between versions/backends.
            nl = DynamicLayer()
            nl.keys = _cl(layer.keys)
            nl.values = _cl(layer.values)
            nl.is_initialized = layer.is_initialized
            nl.dtype = getattr(layer, "dtype", None)
            nl.device = getattr(layer, "device", None)
            for k, v in vars(layer).items():
                if k not in vars(nl):
                    setattr(nl, k, _copy.deepcopy(v))
            new_cache.layers.append(nl)
        else:
            # GDN (LinearAttentionLayer). Clone every state field dynamically
            # rather than hardcoding transformers-5.15.1 attribute names
            # (number_of_states, conv_states, ...): other transformers versions
            # on other backends (e.g. the Kaggle TPU torch 2.8 image) name some
            # differently, and a hardcoded name crashes with AttributeError.
            # Tensors (or dicts of tensors) are clone+detached; scalars and
            # flag-dicts are deep-copied. This is the generic form of the old
            # field-by-field copy and behaves identically on 5.15.1.
            nl = LinearAttentionLayer()
            for k, v in vars(layer).items():
                if isinstance(v, torch.Tensor):
                    setattr(nl, k, _cl(v))
                elif isinstance(v, dict) and v and all(
                    isinstance(x, torch.Tensor) for x in v.values()
                ):
                    setattr(nl, k, {kk: _cl(xv) for kk, xv in v.items()})
                else:
                    setattr(nl, k, _copy.deepcopy(v))
            new_cache.layers.append(nl)
    # Cache object-level state (offloading flag on 5.15.1; 4.x: _seen_tokens):
    # a fresh DynamicCache() would reset it. ``layers`` is rebuilt above;
    # list-valued attrs are skipped to avoid aliasing the original.
    for k, v in vars(cache).items():
        if k == "layers" or isinstance(v, list):
            continue
        setattr(new_cache, k, _copy.deepcopy(v))
    # Guard: the clone must report the same sequence length (loud, not
    # silent, on a version that tracks it in cache state).
    assert new_cache.get_seq_length() == cache.get_seq_length()
    return new_cache
@contextlib.contextmanager
def _clone_gdn_states_on_entry():
    """Clone GDN kernel state inputs at entry: read object != written object.
    transformers 5.15 Qwen3_5GatedDeltaNet.forward resolves the kernels as
    free variables of its own module (modeling_qwen3_5), so patching the
    module attributes works — verified on 5.15.1 (calls at modeling L457/
    L506). On
    a version where GDN forward binds the kernels by a different route
    (an alias imported into another module), the patch silently no-ops and
    the in-place autograd error below resurfaces LOUD — that is the
    detection signal; there is no silent degradation.
    The recurrent loop saves the read object (version 0) for backward.
    Afterwards update_recurrent_state does a stored.copy_(new) IN PLACE on
    the very same object (5.15 cudagraph static-address optimisation) →
    version 0→1 → "modified by an inplace operation" in any backward through
    the decode (create_graph not required; the clone is also a no-op safety
    net for create_graph=True graph retention). Cloning at kernel entry makes
    the loop's saved input the fresh clone; the in-place write lands on the
    cache's original (a different object) → no in-place on a saved node.
    Re-entrancy: nesting is a NO-OP (counter below) — the outer context's
    wrappers are already active and cloning is idempotent (clone-of-clone
    changes nothing). Overwriting mq.<kernel> from OUTSIDE during the patch
    window is user error and is NOT protected (the outer exit restores its
    captured orig, discarding the overwrite).
    If the Qwen3.5 module cannot be imported, a WARNING is logged ONCE per
    interpreter (not per decode; non-Qwen3.5 builds are fine — for Qwen3.5
    with enable_grad=True the in-place autograd error is NOT protected).
    """
    global _GDN_PATCH_DEPTH
    try:
        from transformers.models.qwen3_5 import modeling_qwen3_5 as mq
    except Exception:
        if not getattr(_clone_gdn_states_on_entry, "_warned", False):
            logging.getLogger(__name__).warning(
                "GDN monkey-patch skipped (transformers.models.qwen3_5 not "
                "importable). If the model is Qwen3.5 with enable_grad=True, "
                "in-place autograd errors may occur.")
            _clone_gdn_states_on_entry._warned = True
        yield  # non-Qwen3.5 build: nothing to patch
        return
    _GDN_PATCH_DEPTH += 1
    if _GDN_PATCH_DEPTH > 1:
        # Nested: outer context already patched; wrappers chain, and the
        # clone is idempotent, so nothing to do here.
        yield
        _GDN_PATCH_DEPTH -= 1
        return
    orig_rec = mq.torch_recurrent_gated_delta_rule
    orig_conv = mq.causal_conv1d_update
    def rec(q, k, v, g, beta, *args, initial_state=None, output_final_state=True,
            use_qk_l2norm_in_kernel=False, **kw):
        # initial_state/output_final_state arrive as KEYWORDS in 5.15.1
        # (and may be positional on other versions, hence *args); clone the
        # bound state, forward everything else untouched. A kernel whose
        # signature drops these names falls through to **kw and fails loud.
        if initial_state is not None:
            initial_state = initial_state.clone()
        return orig_rec(q, k, v, g, beta, *args, initial_state=initial_state,
                        output_final_state=output_final_state,
                        use_qk_l2norm_in_kernel=use_qk_l2norm_in_kernel, **kw)
    def conv(h, cs, *args, **kw):
        # (cs, weight, [bias, activation], ...) — positional tail is
        # version-dependent; forward everything we don't own.
        if cs is not None:
            cs = cs.clone()
        return orig_conv(h, cs, *args, **kw)
    mq.torch_recurrent_gated_delta_rule = rec
    mq.causal_conv1d_update = conv
    try:
        yield
    finally:
        mq.torch_recurrent_gated_delta_rule = orig_rec
        mq.causal_conv1d_update = orig_conv
        _GDN_PATCH_DEPTH -= 1
_GDN_PATCH_DEPTH = 0
def _run_decoder_pass(
    model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """Decode h through remaining layers using prefix cache."""
    # Clone cache tensors independently to avoid in-place mutation
    cache_clone = _deep_clone_cache(prefix_cache)
    # Wrapper-safe: flat CausalLM -> model.model; mm wrapper -> .language_model
    text_model = _lm_module(model)
    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)
    text_position_ids = pos[0]
    rope_position_ids = pos[1:]
    position_embeddings = text_model.rotary_emb(h, rope_position_ids)
    x = h
    layer_types = _layer_types(model)
    n_layers = _n_layers(model)
    start = layer_idx + 1
    causal_mask_mapping = _build_causal_masks(
        text_model, text_position_ids, cache_clone, x
    )
    with _clone_gdn_states_on_entry():
        for i in range(start, n_layers):
            layer_type = layer_types[i]
            attn_mask = causal_mask_mapping[layer_type]
            x = text_model.layers[i](
                x,
                position_embeddings=position_embeddings,
                attention_mask=attn_mask,
                position_ids=text_position_ids,
                past_key_values=cache_clone,
            )
    x = text_model.norm(x)
    logits = model.lm_head(x[:, -1, :])
    return logits
def _build_causal_masks(text_model, text_pos, cache, x):
    """Build causal mask mapping for a single token with cache."""
    from transformers.masking_utils import (
        create_causal_mask,
        create_recurrent_attention_mask,
    )
    dummy_embeds = x.new_zeros(1, 1, text_model.config.hidden_size)
    mask_kwargs = {
        "config": text_model.config,
        "inputs_embeds": dummy_embeds,
        "attention_mask": None,
        "past_key_values": cache,
        "position_ids": text_pos,
    }
    return {
        "full_attention": create_causal_mask(**mask_kwargs),
        "linear_attention": create_recurrent_attention_mask(**mask_kwargs),
    }
def _compute_jacobian_rows(
    logits: torch.Tensor, h: torch.Tensor, token_set: list[int]
) -> torch.Tensor:
    """Compute Jacobian rows via ONE batched backward (is_grads_batched).
    Replaces the per-row autograd.grad loop (K backwards through the
    decoder tail) with a single vmap-style pass: grad_outputs=eye(K)
    gives all K rows at once, 3-15x faster on T4 (review point 5).
    Verified identical to the loop on synthetic linear + nonlinear graphs;
    on the real GDN decoder tail this is an EMPIRICAL property of the
    transformers 5.15 kernels (torch_recurrent_gated_delta_rule / causal
    conv1d update under vmap-style batched backward) — there is no
    per-row fallback, so a transformers upgrade that breaks this shows up
    as a runtime error, not silent degradation.
    """
    K = len(token_set)
    return torch.autograd.grad(
        outputs=logits[0, token_set],
        inputs=h,
        grad_outputs=torch.eye(K, device=h.device, dtype=h.dtype),
        is_grads_batched=True,
        retain_graph=False,
    )[0][:, 0, 0, :]
def softmax_jacobian(J: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    """Gauge-invariant Jacobian: J^(p) = (diag(p) - pp^T) @ J.
    J^(p)_i = p_i * (J_i - J_bar_p),  J_bar_p = sum_k p_k * J_k
    Removes the common-mode shift (gauge) from raw Jacobian.
    """
    J_bar = (p[:, None] * J).sum(dim=0, keepdim=True)  # [1, D]
    return p[:, None] * (J - J_bar)  # [K, D]
def uniform_centered(J: torch.Tensor) -> torch.Tensor:
    """Uniform centering: J - mean(J). Removes average shift."""
    return J - J.mean(dim=0, keepdim=True)
def _last_layer_jacobian(h: torch.Tensor, norm: torch.nn.Module) -> torch.Tensor:
    """Analytical J = d h_norm / d h at the final norm (no backward passes).
    Both Qwen3_5RMSNorm and nn.LayerNorm are handled. J is evaluated at the
    norm INPUT (h), never at h_norm — evaluating the formula at the norm
    output gives a matrix with ~0.87 relative error (no longer LayerNorm-like
    statistics). Qwen3.5's final norm is Qwen3_5RMSNorm:
      h_norm = (h * rsqrt(mean(h^2) + eps) * (1 + w))
      J = r · diag(1+w) · (I - xhat xhat^T / D),  r = rsqrt(mean(h^2)+eps), xhat = r*h
    LayerNorm: J = (I - 1/D - xhat xhat^T / D) / std, rows scaled by w.
    """
    # h may arrive as [1, S, D] (full-sequence hidden_states) or [1, D]/[D].
    # The last-layer READOUT Jacobian is the FINAL-position map (it feeds the
    # final-token logits), so collapse a [1, S, D] to its last position first.
    # BUG this fixes: [1, S, D].reshape(1,-1) -> [1, S*D] -> [S*D, S*D] identity
    # (e.g. [65536, 65536] = 16 GB) -> OOM on GPU / XLA compile-abort on TPU
    # (previously misattributed to an "XLA phantom dot" shape bug).
    if h.dim() == 3:
        h = h[:, -1, :]
    x = h.reshape(1, -1).float()  # [1, D], the norm INPUT (row vector)
    D = x.shape[-1]
    eps = float(getattr(norm, "eps", None) or 1e-5)
    w = getattr(norm, "weight", None)
    dev = x.device
    if getattr(norm, "bias", None) is not None:  # LayerNorm family
        mu = x.mean(dim=-1, keepdim=True)
        std = (x.var(dim=-1, unbiased=False, keepdim=True) + eps).sqrt()
        xhat = (x - mu) / std  # [1, D]
        J = (
            torch.eye(D, device=dev)
            - torch.ones(D, D, device=dev) / D
            - xhat.T @ xhat / D
        ) / std.squeeze(0)
        if w is not None:
            J = J * w.float().squeeze(0).unsqueeze(1)  # J = diag(w) @ J
        return J
    # RMSNorm family (Qwen3_5RMSNorm, Llama-style).
    r = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)
    xhat = x * r  # [D]
    # y_j = (1+w_j) r x_j; J = r * diag(1+w) @ (I - xhat xhat^T / D)
    # (derived: J_ij = g_j (r d_ij - r^3 x_i x_j / D) = g_j r (d_ij - xhat_i xhat_j / D))
    g = 1.0 + w.float().squeeze(0) if w is not None else None
    M = torch.eye(D, device=dev) - xhat.T @ xhat / D
    if g is None:
        return r.squeeze(0) * M
    return r.squeeze(0) * torch.diag(g) @ M
def compute_hidden_jacobian(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    create_graph: bool = False,
    verbose: bool = False,
    h: torch.Tensor | None = None,
) -> torch.Tensor:
    """Compute Jacobian of normalized last hidden state w.r.t. hidden state at layer_idx.
    J_norm = d(h_norm) / d(h_{layer_idx}), shape [D, D].
    Independent of lm_head — computed once, reused across W updates.
    Then J_S = lm_head.weight[token_set] @ J_norm is differentiable through W.
    For last_layer: J_norm is LayerNorm Jacobian (analytical, no backward passes).
    Args:
        create_graph: if True, J_norm has grad_fn.
            - If h is None: h is extracted from a full forward and stays
              connected to model params (full gradient flow, as before).
            - If h is given: h is used AS-IS instead of the forward (the
              forward is skipped). A ready leaf h (requires_grad=True,
              grad_fn=None) is used IN PLACE — its .grad receives the
              backpropagated derivative d J_norm/d h. J_norm still depends
              on the decoder WEIGHTS between layer_idx and the final norm
              (the decode pass), so gradients also reach those params; only
              the params of layers <= layer_idx / embeddings are excluded.
                      Costs ~D backward passes per sample.
        verbose: print progress during J_norm computation.
        h: pre-computed hidden state at layer_idx (reuse from a prior forward,
            e.g. KL-forward in exp17). Respected in BOTH create_graph paths
            (v6.0 bug-fix: the create_graph=True path previously ignored it
            and unconditionally re-extracted h from a full forward). Not
            renamed to h_override for backward compatibility; the asymmetry
            with compute_jacobian_for_tokens(h_override=...) is intentional.
    """
    model.eval()
    n_layers = _n_layers(model)
    if create_graph:
        # Keep gradient graph alive: h is connected to model params (+ any hooks)
        S = input_ids.shape[1]
        # Build prefix cache FIRST — before any forward involving hooks.
        prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
        if h is None:
            # Full forward to get h at layer_idx — connected to all model params + hooks
            with torch.enable_grad():
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    output_hidden_states=True,
                    use_cache=False,
                )
            assert outputs.hidden_states is not None  # requested
            h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D], connected!
        # Normalize [1, D] / [D] -> [1, 1, D]; keep any existing grad graph intact.
        h = h.reshape(1, 1, -1) if h.dim() in (1, 2) else h
        model_dtype = next(model.parameters()).dtype
        assert h is not None  # narrowed: param was Tensor|None
        h = h.to(model_dtype)  # preserve grad graph
        if h.grad_fn is None:
            if h.requires_grad:
                # Caller passed a ready leaf that already requires grad
                # (e.g. a representation-holonomy h_leaf). Use it IN PLACE:
                # detach() would create a NEW leaf and the caller's tensor
                # would never receive .grad. It is already a leaf (grad_fn
                # None), so no graph surgery is needed.
                pass
            else:
                # Fully frozen graph: no trainable path reaches this layer
                # (e.g. linear_attention pair with no LoRA patch, or a
                # no_grad caller), OR a disconnected h was passed in. Make h
                # a leaf so the decoder pass below still yields a
                # differentiable h_norm; J_norm values are identical to the
                # connected case.
                h = h.detach().requires_grad_(True)
        if layer_idx == n_layers - 1:
            with torch.enable_grad():
                h_norm = _lm_module(model).norm(h[:, -1, :])  # [1, D]
        else:
            with torch.enable_grad():
                h_norm = _run_decoder_to_norm_compat(
                    model, prefix_cache, h, S, layer_idx
                )
        # [1, 1, D] — _compute_jacobian_rows_full indexes grad[0, 0, :].
        # enable_grad: autograd.grad needs grad mode ON even when the
        # caller is inside torch.no_grad().
        with torch.enable_grad():
            J_norm = _compute_jacobian_rows_full(
                h_norm, h, create_graph=True, verbose=verbose
            )
        return J_norm
    # Last layer: analytical norm Jacobian, no backward passes.
    if layer_idx == n_layers - 1:
        with torch.no_grad():
            if h is None:
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    output_hidden_states=True,
                    use_cache=False,
                )
                assert outputs.hidden_states is not None  # requested
                h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D]
            assert h is not None
            h = h.to(next(model.parameters()).dtype)
        return _last_layer_jacobian(h, _lm_module(model).norm)
    # General case: decode through remaining layers + norm
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    if h is None:
        h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)
    model_dtype = next(model.parameters()).dtype
    h = h.to(model_dtype).detach().requires_grad_(True)
    # Re-enable grad: h_norm must carry a grad_fn over h even if the caller
    # is inside torch.no_grad() (e.g. eval loops in exp16r/exp17). The row
    # loop below (autograd.grad) ALSO needs grad mode ON — a no_grad caller
    # otherwise gets "element 0 ... does not have a grad_fn".
    with torch.enable_grad():
        h_norm = _run_decoder_to_norm_compat(model, prefix_cache, h, S, layer_idx)
        J_norm = _compute_jacobian_rows_full(
            h_norm, h, create_graph=create_graph, verbose=verbose
        )
    return J_norm
def _run_decoder_to_norm(
    text_model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """Decode h through remaining layers, return norm(x) without lm_head."""
    cache_clone = _deep_clone_cache(prefix_cache)
    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)
    text_position_ids = pos[0]
    rope_position_ids = pos[1:]
    position_embeddings = text_model.rotary_emb(h, rope_position_ids)
    x = h
    n_layers = text_model.config.num_hidden_layers
    start = layer_idx + 1
    causal_mask_mapping = _build_causal_masks(
        text_model, text_position_ids, cache_clone, x
    )
    with _clone_gdn_states_on_entry():
        for i in range(start, n_layers):
            layer_type = text_model.config.layer_types[i]
            attn_mask = causal_mask_mapping[layer_type]
            x = text_model.layers[i](
                x,
                position_embeddings=position_embeddings,
                attention_mask=attn_mask,
                position_ids=text_position_ids,
                past_key_values=cache_clone,
            )
    return text_model.norm(x[:, -1, :])
def _compute_jacobian_rows_full(
    output: torch.Tensor,
    h: torch.Tensor,
    create_graph: bool = False,
    verbose: bool = False,
) -> torch.Tensor:
    """Full [D, D] Jacobian via autograd.grad."""
    import time as _time
    D = h.shape[-1]
    out_flat = output.flatten()
    grads = []
    # create_graph=True: every row must retain the graph — the caller
    # backpropagates a final loss through J_norm afterwards (exp17 bridge
    # loss -> o_proj HVP). retain=(i<D-1) would free the (second-order)
    # graph after the last row, so the caller's first backward would fail
    # with "backward through the graph a second time". Memory is bounded
    # by the refresh schedule (one graph per si, freed at the next refresh
    # / pair end).
    start = _time.time()
    for i in range(D):
        grad = torch.autograd.grad(
            outputs=out_flat[i],
            inputs=h,
            retain_graph=(True if create_graph else i < D - 1),
            create_graph=create_graph,
        )[0]
        grads.append(grad[0, 0, :])
        if verbose and i % 128 == 0:
            elapsed = _time.time() - start
            rate = i / elapsed if elapsed > 0 else 999
            eta = elapsed * (D / i) - elapsed if i > 0 else 0.0
            print(f"        J_norm: {i}/{D} ({rate:.0f}/s, eta={eta:.0f}s)")
    J = torch.stack(grads)
    elapsed_total = _time.time() - start
    if verbose:
        print(f"        J_norm done: {D}/{D} in {elapsed_total:.1f}s")
    return J
def _lm_module(model):
    """The text model with .layers/.norm/.rotary_emb.
    Flat CausalLM: model.model IS the text model. Multimodal wrapper
    (Qwen3_5ForConditionalGeneration): model.model.language_model.
    """
    mm = model.model
    if hasattr(mm, "layers"):
        return mm
    return mm.language_model
def _n_layers(model) -> int:
    """num_hidden_layers for flat or wrapper configs."""
    cfg = model.config
    return int(
        cfg.num_hidden_layers
        if hasattr(cfg, "num_hidden_layers")
        else cfg.text_config.num_hidden_layers
    )
def _layer_types(model) -> list:
    """layer_types for flat or wrapper configs."""
    cfg = model.config
    return list(
        cfg.layer_types if hasattr(cfg, "layer_types") else cfg.text_config.layer_types
    )
def _run_decoder_to_norm_compat(
    model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """_run_decoder_to_norm over _lm_module (flat or nested model)."""
    return _run_decoder_to_norm(_lm_module(model), prefix_cache, h, S, layer_idx)
def h_norm_at_perturbed_h(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    h: torch.Tensor,
) -> torch.Tensor:
    """h_norm after the prefix, with the l_S hidden state pinned to `h`.
    Finite-diff support (B2 spot check): call twice, h and h + eps*e_j,
    and difference the two h_norms. h shape [1, 1, D] (any device/dtype);
    the pinning is done on a copy — the input is not mutated.
    """
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    with torch.no_grad():
        return _run_decoder_to_norm_compat(model, prefix_cache, h, S, layer_idx)
def _forward_h_norm(
    model,
    layer_idx,
    input_ids,
    attention_mask,
    h: torch.Tensor | None = None,
):
    """Forward once -> (h, h_norm), h graph-connected at layer l_S.
    Shared by compute_hidden_jacobian_rows and bridge_rows_backward (A4):
    the prefix pass runs before the model forward (hook-safe), h is the
    l_S hidden state (kept in graph so a caller that patched o_proj / lm_head
    gets HVP/bridge grads from the downstream backward), h_norm = RMSNorm(h)
    (last layer) or decoder-to-norm (otherwise).
    h (optional): reuse a pre-computed l_S hidden state instead of doing a
    fresh full forward (v23.1: exp17's KL forward already produced it, so
    the refresh reuses it and only runs the cheap decoder-tail-to-norm).
    """
    model.eval()
    n_layers = _n_layers(model)
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    with torch.enable_grad():
        if h is None:
            # standalone: full forward to obtain h (original A4 behavior)
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                use_cache=False,
            )
            assert outputs.hidden_states is not None  # requested
            h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D]
        assert h is not None
        h = h.to(next(model.parameters()).dtype)
        if h.grad_fn is None:
            h = h.detach().requires_grad_(True)
        if layer_idx == n_layers - 1:
            h_norm = _lm_module(model).norm(h[:, -1, :])  # [1, D]
        else:
            h_norm = _run_decoder_to_norm_compat(
                model, prefix_cache, h, S, layer_idx
            )
    return h, h_norm
def bridge_rows_backward(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
    p: torch.Tensor,
    W_rows: torch.Tensor,
    J_tgt_p: torch.Tensor,
    extra_loss: torch.Tensor | None = None,
    chunk_size: int = 8,
    verbose: bool = False,
    h: torch.Tensor | None = None,
) -> tuple[torch.Tensor, float]:
    """v23 A4: chunked bridge backward (the v22 OOM fix).
    Forward once + J_bar once, then build each chunk of K rows and backprop
    its (1 - cos) bridge loss immediately (extra_loss — the KL — is
    backproped with the first chunk). Backproping per chunk frees that
    chunk's 2nd-order row-graphs at once, so peak = chunk_size + 1 (the
    shared J_bar) second-order graphs, not K. Per-chunk gradients are
    identical to a single backward on the total (1 - cos).mean() loss.
    Gradients reach W_rows (live lm_head + B_lh @ A_lh) and o_proj (HVP
    through h). Returns (J_S_p [K, D] detached, bridge_loss scalar).
    h (optional): reuse a pre-computed l_S hidden state (v23.1) so this
    does NOT redo the full forward — only the cheap decoder-tail-to-norm
    runs. exp17's KL forward already produced the same h.
    """
    import time as _time
    K = len(token_set)
    h, h_norm = _forward_h_norm(
        model, layer_idx, input_ids, attention_mask, h=h
    )
    def _row_grad(w: torch.Tensor) -> torch.Tensor:
        return torch.autograd.grad(
            outputs=h_norm @ w,
            inputs=h,
            retain_graph=True,
            create_graph=True,
        )[0][0, 0, :]
    start = _time.time()
    J_bar = _row_grad(p @ W_rows)  # shared; freed by the last chunk's bwd
    # extra_loss (the KL) gets a SEPARATE backward before the chunk loop.
    # retain_graph=True keeps the shared h-graph alive for the chunks while
    # the KL's OWN forward graph (a separate forward in exp17) frees here.
    # We deliberately do NOT add it to chunk 0: in a shared-forward test/
    # wiring that would free the h-graph the chunks still need.
    if extra_loss is not None:
        extra_loss.backward(retain_graph=True)
    chunks: list[torch.Tensor] = []
    total_bridge = 0.0
    for c in range(0, K, chunk_size):
        hi = min(c + chunk_size, K)
        is_last = (c + chunk_size) >= K
        rows_c = torch.stack(
            [p[i] * (_row_grad(W_rows[i]) - J_bar) for i in range(c, hi)]
        )
        tgt_c = J_tgt_p[c:hi].float()
        cos_c = torch.nn.functional.cosine_similarity(rows_c.float(), tgt_c, dim=1)
        chunk_loss = (1.0 - cos_c).sum() / K
        # retain_graph on all-but-last keeps the ONE shared h forward graph
        # alive so the next chunk can still build its rows; the chunk's
        # 2nd-order row-graph is released by refcounting once we del it
        # (forward nodes stay, referenced by h / h_norm / J_bar).
        chunk_loss.backward(retain_graph=not is_last)
        total_bridge += chunk_loss.item()
        chunks.append(rows_c.detach())
        del rows_c, cos_c, chunk_loss, tgt_c
        if verbose:
            print(
                f"        bridge_rows: {hi}/{K} "
                f"({hi / max(_time.time() - start, 1e-6):.1f} chunks/s)"
            )
    return torch.cat(chunks), total_bridge
def compute_hidden_jacobian_rows(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
    p: torch.Tensor,
    W_rows: torch.Tensor,
    create_graph: bool = False,
    chunk_size: int = 8,
    verbose: bool = False,
) -> torch.Tensor:
    """v23 A1: per-row J_S_p = p_i*(J_i - J_bar), no dense [D,D] J_norm.
    K-subset softmax gauge — the SAME gauge as the exp16 target (p_s =
    softmax(full_logits[token_set])) and the exp17 dense bridge
    (softmax_jacobian(W[token_set]@J_norm, p_L)):
      p = re-normalized dist over the K-subset [K], W_rows = W[token_set]
      [K, D]; J_bar = d(h_norm @ (p @ W_rows))/dh; row_i =
      p[i] * (d(h_norm @ W_rows[i])/dh - J_bar).
    h stays graph-connected (via _run_decoder_to_norm_compat, which
    routes through _lm_module so both flat CausalLM and the multimodal
    wrapper work), so a caller that patched the target layer's o_proj
    and/or lm_head gets its HVP/bridge gradients from the downstream
    loss backward. chunk_size (A4) bounds the 2nd-order graphs alive
    at once; create_graph=True retains them for a later loss backward,
    create_graph=False frees each chunk after the gradient (for cheap
    no-grad refreshes / evals).
    """
    model.eval()
    n_layers = _n_layers(model)
    S = input_ids.shape[1]
    h, h_norm = _forward_h_norm(model, layer_idx, input_ids, attention_mask)
    return _compute_jacobian_rows_matrixfree(
        h_norm,
        h,
        W_rows,
        p,
        token_set,
        create_graph=create_graph,
        chunk_size=chunk_size,
        verbose=verbose,
    )
def _compute_jacobian_rows_matrixfree(
    h_norm: torch.Tensor,
    h: torch.Tensor,
    W_rows: torch.Tensor,
    p: torch.Tensor,
    token_set: list[int],
    create_graph: bool = True,
    chunk_size: int = 8,
    verbose: bool = False,
) -> torch.Tensor:
    """J_S_p [K, D] rows, K-subset softmax gauge, no dense [D,D] J_norm.
    p [K] = re-normalized softmax over the K-subset (exp16/exp17 gauge),
    W_rows [K, D] = the selected rows. J_bar = d(h_norm @ (p @ W_rows))/dh;
    row_i = p[i]*(d(h_norm @ W_rows[i])/dh - J_bar). Identical to
    softmax_jacobian(W_rows @ J_norm, p) but with K backward passes
    instead of a retained [D,D] 2nd-order matrix (v22 OOM).
    Chunked (A4): only chunk_size row-graphs alive at once.
    """
    import time as _time
    K = len(token_set)
    def _row_grad(w: torch.Tensor) -> torch.Tensor:
        return torch.autograd.grad(
            outputs=h_norm @ w,
            inputs=h,
            retain_graph=True,
            create_graph=create_graph,
        )[0][0, 0, :]
    start = _time.time()
    J_bar = _row_grad(p @ W_rows)  # [D]; p@W_rows = effective selected row
    rows = []
    for c in range(0, K, chunk_size):
        for local, i in enumerate(range(c, min(c + chunk_size, K))):
            row = p[i] * (_row_grad(W_rows[i]) - J_bar)
            rows.append(row)
        if verbose:
            done = min(c + chunk_size, K)
            el = _time.time() - start
            print(f"        J_S_p: {done}/{K} ({done / max(el, 1e-6):.1f}/s)")
    out = torch.stack(rows)
    elapsed = _time.time() - start
    if verbose:
        print(f"        J_S_p done: {K}/{K} in {elapsed:.1f}s")
    return out


In [ ]:
import hashlib
import os
import torch
_gramian_cache: dict[str, torch.Tensor] = {}
_GRAMIAN_DIR = os.path.join("results", ".gramians")
def jacobian_gap(
    J_L: torch.Tensor,
    J_S: torch.Tensor,
    E_L: torch.Tensor,
    E_S: torch.Tensor,
    probs_L: torch.Tensor,
    threshold: float = 0.0,
) -> float:
    """Compute the Jacobian gap (alignment loss, LOWER = BETTER) between large and small models.
    Terminology (v4.0): gap = weighted (1 - cos), lower=better; bridge = weighted
    cos, higher=better; gap = 1 - bridge at equal weights. Callers reporting
    "bridge" (E-16a/b) invert: bridge = 1 - jacobian_gap(...). The legacy
    `jacobian_bridge` alias (which silently returned the GAP direction despite
    its name) was removed after being identified as an inversion trap — the
    inversion lives in the experiment kernels (kaggle_kernel/kernel-exp1*/...py),
    not here.
    The bridge chain: J_L @ E_L^T @ E_S @ (E_S^T @ E_S)^{-1}.
    The inverse Gramian (E_S^T @ E_S)^{-1} (disk-cached, see _inverse_gramian)
    corrects for non-orthogonal embeddings, so identity (J_L == J_S,
    E_L == E_S) yields gap ~0.
    threshold: row mask on BOTH norms, (n1 > threshold) & (n2 > threshold).
    Default 0.0 = sentinel-only mode: only exactly-zero rows are masked
    (no decay filtering). Historical default was 1e-6; the paper's
    relative quantile tau_l = 0.01 * median_i ||J_S[i]|| (E-6 calibration,
    ~1e-4 at L18) can be passed explicitly to enable decay masking.
    """
    G_inv = _inverse_gramian(E_S)
    J_L_proj = _project_to_vocab(J_L, E_L)
    J_S_target = _project_to_small(J_L_proj, E_S, G_inv)
    mask = _valid_mask(J_S, J_S_target, threshold)
    if mask.sum() == 0:
        return 1.0
    return _weighted_cosine_loss(J_S, J_S_target, probs_L, mask)
def _project_to_vocab(J_L: torch.Tensor, E_L: torch.Tensor) -> torch.Tensor:
    """Project J_L [K, D_L] through E_L^T -> [K, V]."""
    return J_L.float() @ E_L.float().T
def _project_to_small(
    J_L_proj: torch.Tensor, E_S: torch.Tensor, G_inv: torch.Tensor
) -> torch.Tensor:
    """Project [K, V] through E_S @ G_inv -> [K, D_S].
    G_inv = (E_S^T @ E_S)^{-1} corrects non-orthogonal embeddings.
    """
    return J_L_proj @ E_S.float() @ G_inv
def _inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1}, cached on disk per embedding hash."""
    key = _gramian_key(E)
    if key not in _gramian_cache:
        path = os.path.join(_GRAMIAN_DIR, f"{key}.pt")
        if os.path.exists(path):
            _gramian_cache[key] = torch.load(path, weights_only=True)
        else:
            _gramian_cache[key] = _compute_inverse_gramian(E)
            os.makedirs(_GRAMIAN_DIR, exist_ok=True)
            torch.save(_gramian_cache[key], path)
    return _gramian_cache[key].to(E.device)
def _gramian_key(E: torch.Tensor) -> str:
    """Stable hash of an embedding matrix for disk caching.
    Hashes shape/device header + first, last, and 10 deterministic mid-rows
    (uniformly spaced, seeded from row count so the sample is stable across
    runs but covers the interior). First/last rows alone would collide for
    matrices that share boundary rows (e.g. PAD/UNK) but differ internally.
    Hashing the full 248k×1024 matrix would be prohibitively expensive.
    """
    V = E.shape[0]
    idx = [0, V - 1]
    if V > 4:
        step = (V - 1) // 10
        idx += sorted({(i * step) % (V - 1) + 1 for i in range(1, 10)})
    header = f"{tuple(E.shape)}|{str(E.device)}".encode()
    data = header + E[0].cpu().numpy().tobytes() + E[-1].cpu().numpy().tobytes()
    for i in sorted(set(idx)):
        data += E[i].cpu().numpy().tobytes()
    h = hashlib.sha256(data).hexdigest()[:16]
    return f"g_{h}"
def _compute_inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1} via pseudoinverse for numerical stability."""
    G = E.float().T @ E.float()
    return torch.linalg.pinv(G)
def _valid_mask(
    J_S: torch.Tensor, J_target: torch.Tensor, threshold: float
) -> torch.Tensor:
    """Mask rows where either norm <= threshold (sentinel rows).
    threshold=0.0 (default): only exactly-zero rows are masked.
    threshold>0: decay masking (absolute threshold; the paper's relative
    tau_l = 0.01*median is applied by the caller, E-6 calibration).
    """
    n1 = J_S.float().norm(dim=1)
    n2 = J_target.float().norm(dim=1)
    return (n1 > threshold) & (n2 > threshold)
def _weighted_cosine_loss(
    J_S: torch.Tensor,
    J_target: torch.Tensor,
    probs: torch.Tensor,
    mask: torch.Tensor,
) -> float:
    """Weighted cosine dissimilarity loss."""
    cos_sim = torch.nn.functional.cosine_similarity(
        J_S.float()[mask], J_target[mask], dim=1
    )
    w = probs[mask]
    w = w / (w.sum() + 1e-8)
    return (w * (1.0 - cos_sim)).sum().item()


In [ ]:
import torch
def _validate(jacobians: list[torch.Tensor]) -> None:
    if not jacobians:
        raise ValueError("jacobians: empty list")
    ref = jacobians[0]
    for J in jacobians[1:]:
        if J.shape != ref.shape:
            raise ValueError(f"jacobians: shape mismatch {tuple(J.shape)} vs {tuple(ref.shape)}")
        if J.device != ref.device:
            raise ValueError(f"jacobians: device mismatch {J.device} vs {ref.device}")
def compute_esg(
    jacobians: list[torch.Tensor],
    n_components: int = 32,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute ESG from a list of Jacobian matrices.
    Pooling is INTRA-TRAJECTORY: all J_i must be positions of ONE sample
    (document). Cross-sample pooling is invalid — see module docstring /
    paper §15 (E-0.1); aggregate across samples at the scalar-metric level.
    Args:
        jacobians: list of N Jacobian matrices (one trajectory), each [K, D].
        n_components: number of top eigenvalues/vectors to return.
    Returns:
        eigenvalues: [min(n_components, K)] sorted descending (K < n_components
        possible in degenerate cases; in practice K = |𝒮| ≥ 32).
        eigenvectors: [min(n_components, K), K] corresponding eigenvectors.
    """
    _validate(jacobians)
    G = _build_gramian(jacobians)
    eigenvalues, eigenvectors = _spectral_decomposition(G, n_components)
    return eigenvalues, eigenvectors
def compute_esg_per_sample(
    sample_jacobians: list[list[torch.Tensor]],
    n_components: int = 32,
) -> tuple[list[torch.Tensor], list[torch.Tensor]]:
    """Sample-local ESG: one Gramian per sample, NO cross-sample pooling.
    Args:
        sample_jacobians: list over samples; each entry is a list of the
            sample's per-position Jacobians [K, D].
        n_components: number of top eigenvalues/vectors to return.
    Returns:
        eigenvalues: list [n_samples] of [n_components].
        eigenvectors: list [n_samples] of [n_components, K].
    Cross-sample aggregation happens on scalars (lambda, PR, SNR, r_eff),
    never on the matrices (E-0.1: cross-sample J's are near-orthogonal).
    """
    if not sample_jacobians:
        raise ValueError("sample_jacobians: empty list")
    eigenvalues = []
    eigenvectors = []
    for Js in sample_jacobians:
        ev, evec = compute_esg(Js, n_components=n_components)
        eigenvalues.append(ev)
        eigenvectors.append(evec)
    return eigenvalues, eigenvectors
def cross_positional_gramian(
    J_t: torch.Tensor,
    J_tp: torch.Tensor,
) -> torch.Tensor:
    """Cross-positional Gramian Δ_U(t, t') = J_t @ J_tp.T.
    Args:
        J_t:  [K, D] gauge-free Jacobian at position t.
        J_tp: [K, D] gauge-free Jacobian at position t'.
    Returns:
        [K, K] Gramian (symmetric iff t == t').
    Note:
        - NOT a pooling operation: each call uses exactly two positions.
          See module docstring for intra-trajectory pooling (compute_esg)
          and cross-sample pooling validity (E-0.1).
        - When t == t', result equals the inner-product part of
          compute_esg([J_t]) for N=1: compute_esg([J_t])[0][0] ==
          eigenvalue_0(Δ_U(t, t)). Checked in selfcheck.
        - Caller must ensure J_t, J_tp come from the same model / edit /
          token_set; the function does not check semantic compatibility,
          only shape / device / dtype (via _validate).
    """
    _validate([J_t, J_tp])
    return J_t.float() @ J_tp.float().T
def _build_gramian(jacobians: list[torch.Tensor]) -> torch.Tensor:
    """G = (1/N) * sum_i (J_i @ J_i^T)  (intra-trajectory pool)."""
    U = jacobians[0].shape[0]
    device = jacobians[0].device
    G = torch.zeros(U, U, device=device, dtype=torch.float)
    for J in jacobians:
        G += J.float() @ J.float().T
    G /= len(jacobians)
    return G
def _spectral_decomposition(
    G: torch.Tensor, n_components: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Eigen decomposition, sorted descending."""
    eigenvalues, eigenvectors = torch.linalg.eigh(G)
    # Sort descending
    idx = eigenvalues.argsort(descending=True)
    eigenvalues = eigenvalues[idx][:n_components]
    eigenvectors = eigenvectors[:, idx][:, :n_components].T
    return eigenvalues, eigenvectors


In [ ]:
import warnings
from collections import Counter
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
def uniform_layer_mapping(n_layers_large: int, n_layers_small: int) -> dict[int, int]:
    """Uniform layer mapping from large to small model layers."""
    mapping = {}
    for i in range(n_layers_large):
        j = round(i * (n_layers_small - 1) / (n_layers_large - 1))
        mapping[i] = j
    return mapping
def get_layer_types(model: AutoModelForCausalLM) -> list[str]:
    """Get layer types from model config (full_attention / linear_attention)."""
    cfg = model.config
    return list(
        cfg.layer_types if hasattr(cfg, "layer_types") else cfg.text_config.layer_types
    )
_DOMAIN_DATASETS = {
    "wiki": ("allenai/c4", "en", "train", "text"),
    "math": ("nvidia/OpenMathInstruct-2", None, "train", "problem"),
    "code": ("neyralabs/apex-agents", None, "train", "prompt"),
    "chat": ("HuggingFaceH4/ultrachat_200k", None, "train_sft", "messages"),
}
def _load_dataset_for_domain(
    domain: str,
) -> tuple[str, str | None, str, str]:
    """Return (dataset_id, config, split, text_field) for a domain.
    Falls back to wiki (C4) if the target dataset is unavailable, with a
    RuntimeWarning: a fallback run silently poisons cross-domain comparisons
    (E-13/E-27 T_domain), so the substitution must be visible in the log.
    """
    ds_id, cfg, split, field = _DOMAIN_DATASETS[domain]
    try:
        kw: dict[str, object] = {"split": split, "streaming": True}
        if cfg is not None:
            kw["name"] = cfg
        ds = load_dataset(ds_id, **kw)  # type: ignore[call-overload]
        next(iter(ds))  # force the first fetch: streaming is lazy, a broken/
        # gated dataset would otherwise pass the check and fail inside
        # load_calibration_data (loud crash instead of the intended fallback).
        return ds_id, cfg, split, field
    except Exception:
        warnings.warn(
            f"domain {domain} ({ds_id}) unavailable — falling back to wiki/C4. "
            f"Domain labels in this run are INVALID for cross-domain "
            f"comparisons (E-13/E-27).",
            RuntimeWarning,
            stacklevel=2,
        )
        return _DOMAIN_DATASETS["wiki"]
def _extract_text_from_sample(
    sample: dict,
    field: str,
) -> str:
    """Extract text from a dataset sample, handling different formats."""
    if field == "messages":
        # Chat format: list of {role, content}
        texts = [m.get("content", "") for m in sample.get("messages", [])]
        return " ".join(texts)
    val = sample.get(field, "")
    if isinstance(val, list):
        # e.g. code_contests solutions: list of strings
        return "\n".join(str(v) for v in val)
    if isinstance(val, dict):
        return str(val)
    return val
def load_calibration_data(
    model_name: str,
    max_length: int = 64,
    min_length: int = 64,
    n_samples: int = 4,
    domain: str = "wiki",
    save_offset_mapping: bool = False,
) -> list[dict[str, torch.Tensor]]:
    """Load and tokenize calibration data.
    Defaults match the tested protocol (paper §12/C.6: S=64, filter >= 64,
    N=4); experiments pass explicit lengths. domain: 'wiki' (C4), 'math'
    (OpenMathInstruct-2), 'code' (apex-agents), 'chat' (ultrachat). Falls back
    to wiki if the target dataset is unavailable. save_offset_mapping: adds
    `offset_mapping` to each sample (E-31 sentence boundaries).
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    ds_id, cfg, split, field = _load_dataset_for_domain(domain)
    kw: dict[str, object] = {"split": split, "streaming": True}
    if cfg is not None:
        kw["name"] = cfg
    dataset = load_dataset(ds_id, **kw)  # type: ignore[call-overload]
    samples = []
    for raw in dataset:
        text = _extract_text_from_sample(raw, field)
        if not text:
            continue
        if save_offset_mapping:
            toks = tokenizer(
                text, max_length=max_length, truncation=True,
                add_special_tokens=False, return_offsets_mapping=True,
            )
            if len(toks["input_ids"][0]) >= min_length:
                samples.append({
                    "input_ids": toks["input_ids"][0],
                    "offset_mapping": torch.tensor(
                        toks["offset_mapping"][0], dtype=torch.int32
                    ),
                })
        else:
            tokens = tokenizer(
                text, max_length=max_length, truncation=True,
                return_tensors="pt",
            )
            if tokens["input_ids"].shape[1] >= min_length:
                samples.append(tokens)
        if len(samples) >= n_samples:
            break
    return samples
def build_global_token_set(
    model: AutoModelForCausalLM,
    calibration_samples: list[dict[str, torch.Tensor]],
    K: int = 64,
) -> list[int]:
    """Build global token set from top-K logits across all samples."""
    token_counter: Counter[int] = Counter()
    model.eval()  # type: ignore[attr-defined]
    device = next(model.parameters()).device  # type: ignore[attr-defined]
    with torch.no_grad():
        for sample in calibration_samples:
            inputs = {
                "input_ids": sample["input_ids"].to(device),
                "attention_mask": sample["attention_mask"].to(device),
            }
            outputs = model(**inputs)  # type: ignore[operator]
            logits = outputs.logits[:, -1, :]
            top_idx = logits.topk(K, dim=-1).indices[0]
            token_counter.update(top_idx.tolist())
    U = [tid for tid, _ in token_counter.most_common(K)]
    return sorted(U)
def load_models(
    large_name: str, small_name: str, dtype: torch.dtype = torch.float32
) -> tuple[torch.nn.Module, torch.nn.Module]:
    """Load large and small models with eager attention.
    dtype defaults to float32: the tested protocols run on CPU, where fp16
    is ~10x slower (paper §12/C.9). Pass torch.float16 explicitly for GPU.
    Thread count (torch.set_num_threads(4) in §12/C.9) is set by the caller's
    setup, not here.
    """
    kwargs = {
        "torch_dtype": dtype,
        "attn_implementation": "eager",
    }
    large = AutoModelForCausalLM.from_pretrained(large_name, **kwargs)
    large.eval()  # type: ignore[attr-defined]
    small = AutoModelForCausalLM.from_pretrained(small_name, **kwargs)
    small.eval()  # type: ignore[attr-defined]
    return large, small
def extract_embeddings(
    model: AutoModelForCausalLM,
) -> torch.Tensor:
    """Extract embedding matrix from model.
    The bridge projects through the AN-embedding matrix W (§3/§5). For the
    tied-weight checkpoints in scope, W = E (embed_tokens). If a checkpoint
    is loaded WITHOUT tied weights, W = lm_head.weight ≠ E and the projection
    would silently use the wrong matrix. We therefore warn ONCE per model if
    lm_head.weight is not the same storage as embed_tokens.weight — a loud,
    non-fatal signal that the W = E assumption does not hold for this model.
    """
    emb = model.model.embed_tokens.weight.detach()  # type: ignore[attr-defined]
    _warn_if_not_tied(model, emb)
    return emb
def _warn_if_not_tied(model: AutoModelForCausalLM, emb: torch.Tensor) -> None:
    lm_head = getattr(getattr(model, "lm_head", None), "weight", None)
    if lm_head is None:
        return
    if not lm_head.data_ptr() == emb.data_ptr():
        # data_ptr identity covers tied weights; fall back to value equality
        # for the (rare) case of a tied-but-copied view.
        if not torch.equal(lm_head.detach(), emb):
            warnings.warn(
                f"extract_embeddings: lm_head.weight is NOT tied to "
                f"embed_tokens.weight (model={type(model).__name__}). The "
                f"bridge projects through E assuming W = E (§3); with untied "
                f"weights the an-embedding matrix is lm_head.weight and the "
                f"gap/bridge numbers are silently wrong. Extract lm_head "
                f"explicitly if this is intended.",
                UserWarning,
                stacklevel=2,
            )


In [ ]:
import os
# Max depth under input_root to search (producer puts cache at depth 2:
# results/targets; 4 covers any reasonable layout).
_MAX_DEPTH = 4
def _has_pt(d: str) -> bool:
    try:
        return any(f.endswith(".pt") for f in os.listdir(d))
    except OSError:
        return False
def _walk(root: str, depth: int):
    """Yield dirs under root (listdir-based), depth-limited."""
    try:
        entries = sorted(os.listdir(root))
    except OSError:
        return
    for e in entries:
        p = os.path.join(root, e)
        if _has_pt(p):
            yield p
        if depth > 1:
            yield from _walk(p, depth - 1)
def find_targets_dir(preferred: str, input_root: str = "/kaggle/input") -> str:
    """Return the dir holding the target-cache *.pt files.
    Order: `preferred` (local path / runner-injected), then a listdir walk
    of every mount under `input_root`. Raises FileNotFoundError with the
    mount inventory when nothing is found (no silent fallback, no listdir crash).
    """
    if _has_pt(preferred):
        return preferred
    for hit in _walk(input_root, _MAX_DEPTH):
        return hit
    mounts = []
    try:
        for d in sorted(os.listdir(input_root)):
            try:
                mounts.append(f"{d}/:{os.listdir(os.path.join(input_root, d))[:4]}")
            except OSError:
                mounts.append(f"{d}/:<unreadable>")
    except OSError:
        mounts.append("<no input mount>")
    raise FileNotFoundError(
        f"No target cache .pt in {preferred} nor under {input_root}/ "
        f"(mounts: {mounts}). Run Stage 0 first — it must publish results/targets/*.pt."
    )


In [ ]:
from __future__ import annotations
def _uniform_layer_mapping(n_L: int, n_S: int) -> dict[int, int]:
    """Uniform large->small layer mapping: l_L -> round(l_L*(n_S-1)/(n_L-1)).
    Inlined copy of prism.calibration.uniform_layer_mapping (identical formula)
    so this module stays standalone for kernel inlining.
    """
    return {i: round(i * (n_S - 1) / (n_L - 1)) for i in range(n_L)}
def get_layer_types(model) -> list[str]:
    """Per-layer attention type from model.config.layer_types.
    Qwen3.5 style hybrid: entries 'full_attention' or 'linear_attention'.
    `model.config` (post-AutoModel load) exposes `layer_types` at top level
    (Qwen3_5TextConfig). Returns lowercase list.
    """
    types = getattr(model.config, "layer_types", None)
    if types is None:
        # Fallback: nested text_config (defensive; Qwen3_5 exposes top-level).
        tc = getattr(model.config, "text_config", None)
        types = getattr(tc, "layer_types", None)
    if types is None:
        raise RuntimeError(
            "model.config has no layer_types (and no text_config.layer_types)"
        )
    return [str(t) for t in types]
def select_pairs(n_L: int, n_S: int, layer_types: list[str]) -> list[tuple[int, int]]:
    """Return (l_L, l_S) pairs: full-attn l_S + 0_0 + last, sorted by l_S."""
    if n_L < 1 or n_S < 1 or len(layer_types) != n_S:
        raise ValueError(
            f"invalid: n_L={n_L}, n_S={n_S}, layer_types={len(layer_types)}"
        )
    mapping = _uniform_layer_mapping(n_L, n_S)
    full_S = {i for i, t in enumerate(layer_types) if t == "full_attention"}
    selected = {0, n_S - 1} | full_S
    pairs = [_nearest_preimage(mapping, j) for j in sorted(selected)]
    if (n_L - 1, n_S - 1) not in pairs:
        pairs.append((n_L - 1, n_S - 1))
    return pairs
def _nearest_preimage(mapping: dict[int, int], j: int) -> tuple[int, int]:
    """l_L = argmin |mapping[l] - j|, tie -> smaller l (the selected l_S is j)."""
    best_l, best_d = 0, float("inf")
    for li in range(len(mapping)):
        d = abs(mapping[li] - j)
        if d < best_d:
            best_l, best_d = li, d
    return (best_l, j)
def classify_pair(l_S: int, layer_types: list[str]) -> str:
    """'bridge' if small-model layer l_S has o_proj (full_attention), else 'kl_only'."""
    return (
        "bridge"
        if l_S < len(layer_types) and layer_types[l_S] == "full_attention"
        else "kl_only"
    )


In [ ]:
import gc
import json
import os
import re
import time
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM  # type: ignore[import-not-found]
# resolve_model is provided by build_kernel.py runner cell on Kaggle
resolve_model = lambda hf_id: hf_id  # noqa: E731  # stub for local type checking
def _gd_steps(device: str) -> int:
    return 1000 if str(device).startswith(("cuda", "xla")) else 40
def _eval_every(device: str) -> int:
    return 1 if str(device).startswith(("cuda", "xla")) else 10
def _snap_every(device: str) -> int:
    """Steps between ΔW snapshots (38h/38i need ΔW(s) at dose fractions)."""
    return 10
def main_16readoutpull(
    small_model: str = "Qwen/Qwen3.5-0.8B",
    device: str = "auto",
    max_length: int = 64,
    min_length: int = 32,
    targets_dir: str = "results/targets",
    results_dir: str = "results",
    lr: float = 1e-3,
    kl_weight: float = 0.1,
    knee_tol: float = 0.001,
    steps: int | None = None,
    eval_every: int | None = None,
    samples_from_targets: bool = False,
):
    torch.manual_seed(42)
    if str(device).startswith(("cuda", "xla")):
        gc.collect()
        torch.cuda.empty_cache()
    start = time.time()
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        torch.set_num_threads(4)
    print(f"Device: {device}")
    if steps is None:
        steps = _gd_steps(device)
    if eval_every is None:
        eval_every = _eval_every(device)
    os.makedirs(results_dir, exist_ok=True)
    # -- Load small model -------------------------------------------------
    model_path = small_model
    from contextlib import suppress
    with suppress(NameError):
        model_path = resolve_model(small_model)  # noqa: F821
    print(f"Loading small model: {model_path}")
    mS = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float32,
        attn_implementation="eager",
        low_cpu_mem_usage=True,
    ).to(device)  # type: ignore[arg-type]
    mS.eval()
    n_S = mS.config.num_hidden_layers
    last_layer = n_S - 1
    print(f"Small model: {n_S} layers, last={last_layer}")
    # -- Resolve target cache location ------------------------------------
    # FUSE-safe, layout-independent: preferred dir, else walk /kaggle/input
    # (prism.cache_io — isdir lies on the mount; listdir-only, any slug/depth).
    targets_dir = find_targets_dir(targets_dir)
    print(f"Using target cache at {targets_dir}")
    # -- Load target cache ------------------------------------------------
    cache_files = sorted(
        f
        for f in os.listdir(targets_dir)
        if f.endswith(".pt") and not f.startswith(".")
    )
    if not cache_files:
        raise FileNotFoundError(
            f"No target cache .pt files found at {targets_dir}. Run Stage 0 first."
        )
    cache_by_pair: dict[str, list[dict]] = {}
    for cf in cache_files:
        data = torch.load(os.path.join(targets_dir, cf), weights_only=True)
        pair = data["pair"]
        m = re.search(r"_(\d+)\.pt$", cf)
        data["_si"] = int(m.group(1)) if m else -1
        cache_by_pair.setdefault(pair, []).append(data)
    for v in cache_by_pair.values():
        v.sort(key=lambda t: t["_si"])
    if samples_from_targets:
        # OFFLINE mode (no HF network, e.g. local smoke): the cached
        # input_ids ARE the calibration samples (stage 0 was built from
        # them). One synthetic sample per sample index, sharing the
        # target pair's ids (pairing trivially exact; offline smoke's
        # only purpose is mechanics, not science).
        t0 = list(cache_by_pair.values())[-1][0]
        n_samples = min(2 if not str(device).startswith(("cuda", "xla")) else 4,
                        len(list(cache_by_pair.values())[-1]))
        samples = [
            {
                "input_ids": t0["input_ids"].clone(),
                "attention_mask": torch.ones_like(t0["input_ids"]),
            }
            for _ in range(n_samples)
        ]
        print(f"  (offline: {n_samples} samples from target cache input_ids)")
    else:
        n_samples = 2 if not str(device).startswith(("cuda", "xla")) else 4
        print(f"Loading {n_samples} calibration samples...")
        samples = load_calibration_data(
            small_model,
            max_length=max_length,
            min_length=min_length,
            n_samples=n_samples,
        )
    # The target pair MUST be the last-layer pair: this experiment measures
    # the last-layer readout Jacobian (J_S = W[token_set] @ J_norm,
    # J_norm = ∂h_norm/∂h_last). A shallow target is a null target — matched
    # bridge ~0, indistinguishable from a random rotation. The v1 silent
    # fallback to 0_0 is exactly the bug that voided the v1 results: assert.
    target_pair = None
    for p in sorted(cache_by_pair):
        _, ls = p.split("_")
        if int(ls) == last_layer:
            target_pair = p
            break
    if target_pair is None:
        raise ValueError(
            f"No pair with l_S={last_layer} in {targets_dir} "
            f"(have: {sorted(cache_by_pair)}). Re-run stage 0 with the "
            f"last-layer pair included before this experiment."
        )
    targets = cache_by_pair[target_pair]
    token_set = targets[0]["token_set"]
    K = len(token_set)
    print(f"Using pair {target_pair}, {len(targets)} cached samples, K={K}")
    # -- Sample<->target pairing via cached input_ids -----------------------
    # A mismatch means the reloaded sample differs from the one the target
    # was computed on: that sample's bridge would be a random target, so its
    # bridge term is skipped (KL is unaffected — it uses the sample's own
    # orig logits). No silent position-based pairing.
    pairing_ok: list[bool] = []
    for si in range(min(n_samples, len(targets))):
        t = targets[si]
        ids = samples[si]["input_ids"]
        ok = (
            t.get("input_ids") is not None
            and t["input_ids"].shape[0] == ids.shape[0]
            and bool((t["input_ids"].view(-1) == ids.view(-1)).all())
        )
        pairing_ok.append(ok)
        if not ok:
            print(
                f"  WARNING: sample {si} <-> target {target_pair}_{si} "
                f"input_ids mismatch — its bridge term is skipped"
            )
    print(f"Pairing verified: {sum(pairing_ok)}/{len(pairing_ok)} samples")
    # -- Original lm_head weights -----------------------------------------
    lm_head = mS.lm_head
    W_orig = lm_head.weight.data.clone()
    W_norm = W_orig.norm().item()
    # -- Pre-compute original logits on all samples (for KL leash) ---------
    orig_logits_cache: list[torch.Tensor] = []
    with torch.no_grad():
        for s in samples[:n_samples]:
            ids = s["input_ids"].to(device)
            attn = s["attention_mask"].to(device)
            logits = mS(input_ids=ids, attention_mask=attn).logits[:, -1, :]
            orig_logits_cache.append(logits)
    print("Cached original logits for KL leash")
    # -- Baseline battery (ΔW = 0) — the null reference --------------------
    def battery() -> dict[str, float]:
        bridge_vals: list[float] = []
        with torch.no_grad():
            for si in range(min(n_samples, len(targets))):
                if not pairing_ok[si]:
                    continue
                tgt = targets[si]
                s = samples[si]
                ids = s["input_ids"].to(device)
                attn = s["attention_mask"].to(device)
                out = mS(
                    input_ids=ids,
                    attention_mask=attn,
                    output_hidden_states=True,
                    use_cache=False,
                )
                h_s = out.hidden_states[last_layer + 1]
                J_norm = compute_hidden_jacobian(
                    mS, last_layer, ids, attn, h=h_s
                )
                J_S_ed = lm_head.weight[token_set] @ J_norm
                p_L = tgt["p"].to(device)
                J_S_ed_p = softmax_jacobian(J_S_ed, p_L)
                cos = (
                    torch.nn.functional.cosine_similarity(
                        J_S_ed_p.float(), tgt["J_tgt_p"].to(device).float(), dim=1
                    )
                    .mean()
                    .item()
                )
                bridge_vals.append(cos)
        kl_sum = 0.0
        with torch.no_grad():
            for si in range(min(n_samples, len(orig_logits_cache))):
                s = samples[si]
                ids = s["input_ids"].to(device)
                attn_s = s["attention_mask"].to(device)
                logits_ed = mS(input_ids=ids, attention_mask=attn_s).logits[
                    :, -1, :
                ]
                kl = F.kl_div(
                    F.log_softmax(orig_logits_cache[si].float(), dim=-1),
                    F.log_softmax(logits_ed.float(), dim=-1),
                    log_target=True,  # NaN without it (Kaggle run 2026-08-23)
                    reduction="batchmean",
                )
                kl_sum += kl.item()
        return {
            "bridge": sum(bridge_vals) / len(bridge_vals) if bridge_vals else 0.0,
            "KL": kl_sum / min(n_samples, len(orig_logits_cache)),
        }
    baseline = battery()
    print(
        f"Baseline (ΔW=0): bridge={baseline['bridge']:.4f}, "
        f"KL={baseline['KL']:.6f}"
    )
    # -- Single unclamped Adam run with per-eval dose recording ------------
    # The old trust-region buffer block is GONE on purpose: no cap, so
    # ‖ΔW‖ is a real, unbounded dose. (No allocator churn either: the
    # in-place sub() buffer existed only to apply the clamp.)
    # ΔW SNAPSHOT GRID (added for E-38h/38i): the E-16 run edits lm_head
    # in-place and previously published only the scalar dose curve — the
    # ΔW(s) tensors that E-38h/38i need were never on disk. We snapshot
    # the token-set slice ΔW(s) = (W_ed − W_orig)[token_set] (fp16, ~262 KB
    # each) every `snap_every` steps into ONE file, plus W_orig[token_set].
    # Offline, for any target fraction s (0.1/0.25/0.5/0.75/1.0 of final
    # dose) we pick the snapshot with dose closest to s·dose_final. Dose is
    # monotone along the run (verified on the v2 curve), so quantization
    # with snap_every=10 is ≤ ~1% of final dose — well inside 38h/38i needs.
    optimizer = torch.optim.Adam([lm_head.weight], lr=lr)
    snap_every = _snap_every(device)
    snap_dir = os.path.join(results_dir, "exp16_deltas")
    os.makedirs(snap_dir, exist_ok=True)
    snapshots: dict[int, torch.Tensor] = {}
    curve: list[dict] = [
        {"step": 0, "dose": 0.0, **baseline}
    ]
    t0 = time.time()
    for step in range(steps):
        optimizer.zero_grad()
        total_loss_val = 0.0
        for si in range(min(n_samples, len(targets))):
            s = samples[si]
            ids = s["input_ids"].to(device)
            attn = s["attention_mask"].to(device)
            tgt = targets[si]
            # KL leash: true KL(p_ed || p_orig) over the full vocab.
            # log_target=True is MANDATORY: without it PyTorch computes
            # target*(log(target)-input) with target already a log-prob
            # (<=0) -> log(negative) = NaN from step 1.
            out = mS(
                input_ids=ids,
                attention_mask=attn,
                output_hidden_states=True,
                use_cache=False,
            )
            kl_loss = F.kl_div(
                F.log_softmax(orig_logits_cache[si].float(), dim=-1),
                F.log_softmax(out.logits.float(), dim=-1),
                log_target=True,
                reduction="batchmean",
            )
            if pairing_ok[si]:
                # Fresh per-sample J_norm, differentiable through lm_head.
                h_s = out.hidden_states[last_layer + 1]
                J_norm = compute_hidden_jacobian(
                    mS, last_layer, ids, attn, h=h_s
                )
                J_S = lm_head.weight[token_set] @ J_norm  # [K, D]
                p_L = tgt["p"].to(device)
                J_S_p = softmax_jacobian(J_S, p_L)
                cos_sim = torch.nn.functional.cosine_similarity(
                    J_S_p.float(), tgt["J_tgt_p"].to(device).float(), dim=1
                )
                loss = (1.0 - cos_sim).mean() + kl_weight * kl_loss
            else:
                loss = kl_weight * kl_loss
            total_loss_val += loss.item()
            loss.backward()
        optimizer.step()
        if (step + 1) % 5 == 0:
            print(
                f"  step {step + 1}/{steps}: loss={total_loss_val:.4f} "
                f"({(time.time() - t0) / ((step + 1) // 5 + 1):.1f}s/eval-5)"
            )
        if (step + 1) % snap_every == 0 or step + 1 == steps:
            dW_snap = (
                lm_head.weight.data - W_orig
            )[token_set].to(torch.float16).cpu()
            snapshots[step + 1] = dW_snap
        if (step + 1) % eval_every == 0 or step + 1 == steps:
            dose = (lm_head.weight.data - W_orig).norm().item()
            b = battery()
            curve.append(
                {"step": step + 1, "dose": dose, **b}
            )
            print(
                f"  [eval] step {step + 1}: dose ‖ΔW‖={dose:.4f} "
                f"(α={dose / W_norm:.4f}), bridge={b['bridge']:.4f}, "
                f"KL={b['KL']:.6f}"
            )
    del optimizer
    gc.collect()
    if str(device).startswith(("cuda", "xla")):
        torch.cuda.empty_cache()
    # -- Save ΔW snapshot file (E-38h/38i input) ----------------------------
    # One file: {"W_orig": W_orig[token_set] fp16 [K,D], "snapshots":
    # {step: ΔW[token_set] fp16 [K,D]}, "token_set": [...], "dose": {step: ‖ΔW‖}}.
    # fp16 is exact enough: ΔW components are O(0.5), fp16 rel. eps ~1e-3.
    snap_payload = {
        "W_orig": W_orig[token_set].to(torch.float16).cpu(),
        "snapshots": snapshots,
        "token_set": list(token_set),
        "step": steps,
        "final_dose": float(
            (lm_head.weight.data - W_orig).norm()
        ),
    }
    snap_path = os.path.join(snap_dir, "exp16_dW_snapshots.pt")
    torch.save(snap_payload, snap_path)
    print(
        f"Saved {snap_path}: {len(snapshots)} ΔW snapshots "
        f"(final dose {snap_payload['final_dose']:.2f}) for E-38h/38i"
    )
    # -- α* (the knee): largest step t with Δbridge ≥ knee_tol -------------
    # Δbridge(t) = bridge(t) − bridge(t − eval_every) on the same curve —
    # a genuine dose-derivative. If every step improves past tol, no knee
    # exists: report None (honest null), not the first slope.
    alpha_star: float | None = None
    knee_step: int | None = None
    for i in range(1, len(curve)):
        db = curve[i]["bridge"] - curve[i - 1]["bridge"]
        if db >= knee_tol:
            alpha_star = curve[i]["dose"] / W_norm
            knee_step = curve[i]["step"]
    if alpha_star is None:
        print(f"\nα* = null (no knee: bridge kept improving ≥ {knee_tol} "
              f"per eval window to the end)")
    else:
        print(f"\nα* (knee) = {alpha_star:.4f} at step {knee_step}")
    # -- Save lm_head weights at knee + final for inference inspection -------
    # Save only ΔW in fp16 (311MB, downloadable) — NOT W/W_orig (3GB each).
    # ΔW applied to the 0.8B's own lm_head = the "edited" model for inference.
    pt_paths: list[str] = []
    delta_W = (lm_head.weight.data - W_orig).cpu().half()
    pt_final = os.path.join(results_dir, "exp16_lm_head_final.pt")
    torch.save({
        "delta_W": delta_W,
        "step": steps,
        "bridge": curve[-1]["bridge"],
        "dose": float(delta_W.float().norm()),
    }, pt_final)
    pt_paths.append(pt_final)
    print(f"Saved {pt_final} (‖ΔW‖={delta_W.float().norm().item():.4f}, bridge={curve[-1]['bridge']:.4f})")
    # Knee weight (if knee detected)
    if knee_step is not None and knee_step < steps:
        # Re-derive the weight at the knee by running Adam to that step is
        # expensive. Instead: the curve stores dose at each eval point, and
        # we saved the final. For knee, we'd need to have saved it during
        # the loop. Save a marker with the knee dose for re-construction.
        # (The user can re-run with steps=knee_step to get the exact weight.)
        pt_knee = os.path.join(results_dir, "exp16_lm_head_knee_info.pt")
        torch.save({
            "knee_step": knee_step,
            "knee_dose": alpha_star * W_norm,
            "knee_bridge": curve[min(knee_step // max(eval_every, 1), len(curve) - 1)]["bridge"],
            "note": "Run exp16 with --steps <knee_step> to save the exact weight",
        }, pt_knee)
        pt_paths.append(pt_knee)
        print(f"Saved {pt_knee} (knee at step {knee_step}, dose ‖ΔW‖={alpha_star * W_norm:.4f})")
    # -- Save results ------------------------------------------------------
    result = {
        "description": (
            "Readout-only pull as a DOSE-RESPONSE curve (v2). One unclamped "
            "Adam run on lm_head: loss = (1-cos(J_S_p, J_tgt_p)) per "
            "input_ids-paired sample + 0.1·KL(p_ed‖p_orig). Every 10 steps: "
            "dose ‖ΔW‖_F (α = ‖ΔW‖/‖W‖), bridge (fresh J_norm via h-reuse), "
            "true KL over the full vocab. Baseline at ΔW=0 included. α* = "
            "largest step where the per-window bridge gain ≥ knee_tol "
            "(None if no knee). v1 (alpha sweep as trust-region caps) was "
            "VOID: caps ≥0.1 never bind, α was not a dose, and its 'α*=0.01' "
            "was a first-slope artifact. v2 fixes the dose definition. "
            "Target pair is the last-layer pair (asserted)."
        ),
        "curve": curve,
        "baseline": baseline,
        "alpha_star": alpha_star,
        "knee_step": knee_step,
        "knee_tol": knee_tol,
        "lr": lr,
        "kl_weight": kl_weight,
        "steps": steps,
        "eval_every": eval_every,
        "W_norm": W_norm,
        "target_pair": target_pair,
        "pairing": {
            "mode": "input_ids",
            "verified": int(sum(pairing_ok)),
            "total": int(len(pairing_ok)),
        },
        "n_layers_S": n_S,
        "last_layer": last_layer,
        "K": K,
        "device": device,
        "elapsed_seconds": time.time() - start,
        "params": {
            "small_model": small_model,
            "device": device,
            "max_length": max_length,
            "min_length": min_length,
            "targets_dir": targets_dir,
        },
    }
    path = os.path.join(results_dir, "exp16_readout_pull.json")
    with open(path, "w") as f:
        json.dump(result, f, indent=2)
    print(f"\nSaved to {path}")
    print(f"Total time: {time.time() - start:.0f}s")


In [ ]:
# ---------- run exp16_readout_pull ----------
import gc
# Hard cleanup from previous kernel version (same session, different code)
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
import inspect
sig = inspect.signature(main_16readoutpull)
kwargs = {}
params = list(sig.parameters)
if 'small_model' in params: kwargs['small_model'] = SMALL
if 'large_model' in params: kwargs['large_model'] = LARGE
if 'model' in params: kwargs['model'] = SMALL
if 'device_L' in params: kwargs['device_L'] = device_L
if 'device_S' in params: kwargs['device_S'] = device_S
if 'device' in params: kwargs['device'] = device_S
if 'results_dir' in params: kwargs['results_dir'] = RES
if 'targets_dir' in params: kwargs['targets_dir'] = os.path.join(RES, 'targets')
# Remove conflicting keys: if both model and small_model, prefer specific
if 'small_model' in kwargs and 'model' in kwargs: del kwargs['model']
# Memory diagnostic before experiment
if torch.cuda.is_available():
    print(f'cuda:0 alloc={torch.cuda.memory_allocated(0)/1024**3:.1f}GiB reserved={torch.cuda.memory_reserved(0)/1024**3:.1f}GiB')
    print(f'cuda:1 alloc={torch.cuda.memory_allocated(1)/1024**3:.1f}GiB reserved={torch.cuda.memory_reserved(1)/1024**3:.1f}GiB')
t0 = time.time()
try:
    main_16readoutpull(**kwargs)
    elapsed = time.time() - t0
    print(f"[OK] exp16_readout_pull {elapsed:.0f}s")
    # Persist models (in case experiment loaded from HF but didn't save)
    try:
        dst_s = os.path.join(WORKING, "qwen35-08b")
        dst_l = os.path.join(WORKING, "qwen35-4b")
        if os.path.isdir(dst_s):
            print(f"Small model persisted at {dst_s}")
        if os.path.isdir(dst_l):
            print(f"Large model persisted at {dst_l}")
    except Exception:
        pass
except Exception as e:
    print(f"[FAIL] exp16_readout_pull: {e}")
    traceback.print_exc()
print("\nResults:")
if os.path.isdir(RES):
    try:
        for fn in sorted(os.listdir(RES)):
            print(f"  {fn} {os.path.getsize(os.path.join(RES, fn)):,} bytes")
    except OSError:
        pass
